# 02b LSTM-Only Strict Forecasting

Use this notebook when table-model artifacts from `02_table_model_forecasting.ipynb` are already acceptable and you want to iterate only on the per-ticker LSTM. It trains LSTM models into an isolated `lstm_only/strict_protocol` artifact tree, then merges LSTM rows into the main strict reports so `03_model_comparison.ipynb` can compare all models without rerunning table models.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "forecasting", cwd.parent, cwd.parent / "forecasting"]:
    if (candidate / "src" / "stock_forecast").exists():
        PROJECT_DIR = candidate
        break
else:
    raise RuntimeError("Cannot locate forecasting project directory with src/stock_forecast")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "data"
REPORTS_DIR = ARTIFACT_DIR / "reports"
for path in [DATA_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /home/sapce/forecasting_stock_prices/forecasting


In [2]:
import importlib.util
import json
import shutil

import pandas as pd
from IPython.display import display

from stock_forecast.artifacts import load_json, load_table, save_json, save_table
from stock_forecast.models import build_model
from stock_forecast.strict_protocol import (
    run_strict_per_ticker_protocol,
    strict_protocol_leakage_audit,
    _strict_metrics,
    _validation_ranking,
)
from stock_forecast.utils import ensure_dir

pd.set_option("display.max_columns", 180)

## Constants

In [3]:
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is required for this notebook. Install the forecasting[deep] extra.")

FORCE_RETRAIN = True  # set True when you want to retune/retrain LSTM even if matching cache exists
PRIMARY_METRIC = "directional_accuracy"
RANDOM_STATE = 42

STRICT_VALIDATION_ROWS = 126
STRICT_TEST_ROWS = 126
MATURE_MIN_ROWS = 1008
LIMITED_HISTORY_MIN_BLOCK_ROWS = 42
MIN_TRAIN_ROWS = 60
STRICT_MAX_TRAIN_ROWS = 1260
INNER_MAX_FOLDS = 3
INNER_MIN_TRAIN_ROWS = 126
LIMITED_HISTORY_N_TRIALS = 20

LSTM_N_TRIALS = 60
LSTM_OPTUNA_N_JOBS = 1
LSTM_MAX_EPOCHS = 150
LSTM_PATIENCE = 15
LSTM_DEVICE = "auto"
LSTM_ENSEMBLE_SEEDS = [1, 7, 21, 42, 101]

TRANSACTION_COST_BPS = 10
SLIPPAGE_BPS = 5
LONG_THRESHOLD = 0.0
SIGNAL_ANCHOR = "expanding_median"

MERGE_LSTM_INTO_MAIN_STRICT_REPORTS = True
LSTM_ONLY_ARTIFACT_NAME = "lstm_only"

HORIZONS = [
    {"name": "week", "horizon": 5},
    {"name": "month", "horizon": 21},
]

In [4]:
def make_lstm_search_space(horizon: int, limited_history: bool = False) -> dict:
    huber_beta_choices = [0.04, 0.08, 0.12] if horizon >= 21 else [0.02, 0.04, 0.06]
    return {
        "lookback": {"type": "categorical", "choices": [20, 40, 60] if limited_history else [20, 40, 60, 90, 126]},
        "hidden_size": {"type": "categorical", "choices": [16, 32, 64, 96, 128]},
        "num_layers": {"type": "categorical", "choices": [1] if limited_history else [1, 2]},
        "input_projection_size": {"type": "categorical", "choices": [0, 32, 64, 128]},
        "lstm_dropout": {"type": "float", "low": 0.0, "high": 0.35},
        "head_dropout": {"type": "float", "low": 0.15, "high": 0.55},
        "learning_rate": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
        "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
        "batch_size": {"type": "categorical", "choices": [16, 32, 64] if limited_history else [32, 64, 128]},
        "loss": {"type": "categorical", "choices": ["smooth_l1", "huber", "mse"]},
        "huber_beta": {"type": "categorical", "choices": huber_beta_choices},
        "feature_clip": {"type": "categorical", "choices": [3.0, 5.0, 8.0]},
        "grad_clip_norm": {"type": "float", "low": 0.5, "high": 2.0},
    }


def make_lstm_config(lstm_feature_cols: list[str], horizon: int) -> dict:
    return {
        "name": "lstm",
        "model_type": "lstm",
        "estimator_factory": build_model,
        "input_mode": "full_frame",
        "feature_cols": lstm_feature_cols,
        "static_params": {
            "max_epochs": LSTM_MAX_EPOCHS,
            "patience": LSTM_PATIENCE,
            "device": LSTM_DEVICE,
        },
        "search_space": make_lstm_search_space(horizon, limited_history=False),
        "limited_history_search_space": make_lstm_search_space(horizon, limited_history=True),
        "post_selection_static_params": {"ensemble_seeds": LSTM_ENSEMBLE_SEEDS},
        "n_trials": LSTM_N_TRIALS,
        "optuna_n_jobs": LSTM_OPTUNA_N_JOBS,
        "limited_history_n_trials": LIMITED_HISTORY_N_TRIALS,
        "needs_scaler": False,
    }

## Load LSTM Data Artifacts

In [5]:
horizon_inputs = []

for spec in HORIZONS:
    horizon_name = spec["name"]
    horizon = int(spec["horizon"])
    base_feature_payload = load_json(DATA_DIR / "horizons" / horizon_name / "feature_columns.json")
    target_col = base_feature_payload["target_column"]

    lstm_dir = DATA_DIR / "lstm" / "horizons" / horizon_name
    lstm_model_path = lstm_dir / "model_dataset.parquet"
    lstm_feature_path = lstm_dir / "feature_columns.json"
    if not lstm_model_path.exists() and not lstm_model_path.with_suffix(".csv").exists():
        raise FileNotFoundError(
            f"Missing LSTM data for {horizon_name}: {lstm_model_path}. "
            "Run notebooks/01b_lstm_eda.ipynb first."
        )
    if not lstm_feature_path.exists():
        raise FileNotFoundError(f"Missing LSTM feature payload for {horizon_name}: {lstm_feature_path}")

    lstm_payload = load_json(lstm_feature_path)
    if lstm_payload["target_column"] != target_col:
        raise ValueError(f"LSTM target mismatch for {horizon_name}: {lstm_payload['target_column']} != {target_col}")

    model_df = load_table(lstm_model_path)
    model_df["date"] = pd.to_datetime(model_df["date"])
    lstm_feature_cols = lstm_payload["feature_columns"]
    model_configs = [make_lstm_config(lstm_feature_cols, horizon)]

    horizon_inputs.append({
        "horizon_name": horizon_name,
        "horizon": horizon,
        "model_df": model_df,
        "feature_cols": base_feature_payload["feature_columns"],
        "lstm_feature_cols": lstm_feature_cols,
        "target_col": target_col,
        "model_configs": model_configs,
        "lstm_only_artifact_dir": ARTIFACT_DIR / "horizons" / horizon_name / LSTM_ONLY_ARTIFACT_NAME,
        "main_artifact_dir": ARTIFACT_DIR / "horizons" / horizon_name,
    })

    print({
        "horizon": horizon_name,
        "rows": len(model_df),
        "lstm_features": len(lstm_feature_cols),
        "target": target_col,
        "lstm_only_artifact_dir": str((ARTIFACT_DIR / "horizons" / horizon_name / LSTM_ONLY_ARTIFACT_NAME).relative_to(PROJECT_DIR)),
    })
    display(model_df[["date", "ticker", target_col, *lstm_feature_cols[:6]]].head())

{'horizon': 'week', 'rows': 17880, 'lstm_features': 160, 'target': 'target_return_5_next_open', 'lstm_only_artifact_dir': 'artifacts/horizons/week/lstm_only'}


,date,ticker,target_return_5_next_open,log_close,ret_1,open_close_ret,high_low_range,close_to_high,close_to_low
0,2015-11-09,CBOM,0.013316,1.321756,-0.003992,0.000000,0.000000,0.000000,0.000000
1,2015-11-10,CBOM,0.019908,1.320422,-0.001334,0.004013,0.004005,0.000000,0.004021
2,2015-11-11,CBOM,0.005312,1.319086,-0.001336,0.002677,0.002674,0.000000,0.002681
3,2015-11-12,CBOM,0.010582,1.323088,0.004003,0.000000,0.000000,0.000000,0.000000
4,2015-11-13,CBOM,-0.002649,1.329724,0.006636,0.005305,0.300265,-0.227783,0.005319


{'horizon': 'month', 'rows': 17768, 'lstm_features': 160, 'target': 'target_return_21_next_open', 'lstm_only_artifact_dir': 'artifacts/horizons/month/lstm_only'}


,date,ticker,target_return_21_next_open,log_close,ret_1,open_close_ret,high_low_range,close_to_high,close_to_low
0,2015-11-09,CBOM,0.027761,1.321756,-0.003992,0.000000,0.000000,0.000000,0.000000
1,2015-11-10,CBOM,0.023842,1.320422,-0.001334,0.004013,0.004005,0.000000,0.004021
2,2015-11-11,CBOM,0.021081,1.319086,-0.001336,0.002677,0.002674,0.000000,0.002681
3,2015-11-12,CBOM,0.021053,1.323088,0.004003,0.000000,0.000000,0.000000,0.000000
4,2015-11-13,CBOM,0.011834,1.329724,0.006636,0.005305,0.300265,-0.227783,0.005319


## Train LSTM Only

In [6]:
lstm_results = {}

for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    horizon = item["horizon"]
    result = run_strict_per_ticker_protocol(
        model_df=item["model_df"],
        feature_cols=item["feature_cols"],
        target_col=item["target_col"],
        model_configs=item["model_configs"],
        artifact_dir=item["lstm_only_artifact_dir"],
        force_retrain=FORCE_RETRAIN,
        primary_metric=PRIMARY_METRIC,
        random_state=RANDOM_STATE,
        run_metadata={
            "horizon_name": horizon_name,
            "horizon": horizon,
            "training_notebook": "02b_lstm_forecasting",
            "lstm_only_run": True,
        },
        validation_rows=STRICT_VALIDATION_ROWS,
        test_rows=STRICT_TEST_ROWS,
        mature_min_rows=MATURE_MIN_ROWS,
        limited_history_min_block_rows=LIMITED_HISTORY_MIN_BLOCK_ROWS,
        min_train_rows=MIN_TRAIN_ROWS,
        max_train_rows=STRICT_MAX_TRAIN_ROWS,
        inner_max_folds=INNER_MAX_FOLDS,
        inner_min_train_rows=INNER_MIN_TRAIN_ROWS,
        limited_history_n_trials=LIMITED_HISTORY_N_TRIALS,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        slippage_bps=SLIPPAGE_BPS,
        long_threshold=LONG_THRESHOLD,
        signal_anchor=SIGNAL_ANCHOR,
    )
    lstm_results[horizon_name] = result

    print(f"=== LSTM-only strict protocol: {horizon_name} ({horizon} trading days) ===")
    display(result["validation_model_ranking"])
    display(result["test_prediction_metrics"])
    display(result["test_signal_metrics"])
    display(result["leakage_audit"])

    failed = result["leakage_audit"][~result["leakage_audit"]["passed"]]
    if not failed.empty:
        raise AssertionError(f"LSTM-only leakage audit failed for {horizon_name}: {failed['check'].tolist()}")

=== LSTM-only strict protocol: week (5 trading days) ===


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
0,validation,CBOM,lstm,126,0.050201,0.064949,-0.050582,0.017727,0.013129,0.460317,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,1,True
1,validation,MBNK,lstm,64,0.035874,0.047401,-1.229204,0.213218,0.205311,0.656250,week,5,limited_history,True,184,190,6,190,1260,2024-09-05,2025-05-15,1,True
2,validation,SBER,lstm,126,0.028419,0.040303,-0.062120,-0.032598,0.024960,0.515873,week,5,mature,False,1254,1260,6,4158,1260,2019-12-02,2024-12-09,1,True
3,validation,SBERP,lstm,126,0.034334,0.047882,-0.704068,0.120428,0.151940,0.500000,week,5,mature,False,1254,1260,6,4158,1260,2019-12-02,2024-12-09,1,True
4,validation,SVCB,lstm,83,0.042029,0.050334,-0.581120,0.130747,0.230637,0.469880,week,5,limited_history,True,245,251,6,251,1260,2024-04-27,2025-04-03,1,True
5,validation,T,lstm,126,0.044666,0.058931,-0.190015,0.015704,0.016627,0.476190,week,5,mature,False,1162,1168,6,1168,1260,2020-03-13,2024-12-05,1,True
6,validation,VTBR,lstm,126,0.047678,0.065448,-0.100276,0.072973,0.091876,0.515873,week,5,mature,False,1254,1260,6,4114,1260,2019-11-26,2024-12-09,1,True


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,lstm,126,0.032930,0.042199,-0.020033,0.044701,0.079877,0.539683,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,1,True,0.460317
1,test,MBNK,lstm,64,0.027314,0.039385,-0.038791,0.387282,0.447253,0.625000,week,5,limited_history,True,248,254,6,254,1260,2024-09-05,2025-07-23,1,True,0.656250
2,test,SBER,lstm,126,0.016785,0.020871,-0.262785,-0.107531,-0.158221,0.484127,week,5,mature,False,1254,1260,6,4284,1260,2020-06-05,2025-05-17,1,True,0.515873
3,test,SBERP,lstm,126,0.018960,0.023584,-0.621203,-0.069474,-0.081893,0.476190,week,5,mature,False,1254,1260,6,4284,1260,2020-06-05,2025-05-17,1,True,0.500000
4,test,SVCB,lstm,83,0.028785,0.038361,0.024605,0.205835,0.122455,0.698795,week,5,limited_history,True,328,334,6,334,1260,2024-04-27,2025-07-04,1,True,0.469880
5,test,T,lstm,126,0.022514,0.028851,-0.113026,0.035315,0.064411,0.603175,week,5,mature,False,1254,1260,6,1294,1260,2020-04-30,2025-05-15,1,True,0.476190
6,test,VTBR,lstm,126,0.041385,0.069702,-0.055636,0.020533,0.059114,0.563492,week,5,mature,False,1254,1260,6,4240,1260,2020-06-01,2025-05-17,1,True,0.515873


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning
0,0.032482,0.066018,0.112762,252.0,0.585464,0.894060,-0.145398,0.454050,0.014286,9,CBOM,lstm,overlapping_tranches,126,False
1,0.105406,0.483771,0.095615,252.0,5.059591,9.187456,-0.035494,13.629735,0.018750,6,MBNK,lstm,overlapping_tranches,64,False
2,-0.076245,-0.146676,0.044104,252.0,-3.325693,-3.973706,-0.112215,-1.307100,0.022222,14,SBER,lstm,overlapping_tranches,126,False
3,-0.059965,-0.116334,0.058124,252.0,-2.001478,-3.330499,-0.110333,-1.054386,0.030159,19,SBERP,lstm,overlapping_tranches,126,False
4,-0.133801,-0.353455,0.118152,252.0,-2.991526,-4.601191,-0.231166,-1.529013,0.028916,12,SVCB,lstm,overlapping_tranches,83,False
5,-0.011244,-0.022361,0.059983,252.0,-0.372783,-0.426275,-0.096451,-0.231835,0.049206,31,T,lstm,overlapping_tranches,126,False
6,-0.228242,-0.404389,0.211392,252.0,-1.912979,-1.706576,-0.308063,-1.312682,0.020635,13,VTBR,lstm,overlapping_tranches,126,False
7,0.131492,0.270578,0.262557,50.4,1.030551,1.957683,-0.140755,1.922329,0.192308,5,CBOM,lstm,non_overlapping,26,False
8,0.043339,0.178783,0.220048,50.4,0.812473,1.154788,-0.094292,1.896052,0.153846,2,MBNK,lstm,non_overlapping,13,False
9,-0.043073,-0.081807,0.095230,50.4,-0.859040,-1.148620,-0.113110,-0.723244,0.230769,6,SBER,lstm,non_overlapping,26,False


,check,passed,details
0,strict outer splits are available,True,split_rows=7
1,validation predictions are available,True,rows=777
2,test predictions are available,True,rows=777
3,outer split dates are chronological,True,bad_rows=0
4,train and refit windows respect max_train_rows,True,"train_over_cap=0, refit_over_cap=0"
5,validation predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
7,final refit target dates end before test starts,True,overlap_rows=0
8,final model payloads exist for test predictions,True,missing_models=0


=== LSTM-only strict protocol: month (21 trading days) ===


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
0,validation,CBOM,lstm,126,0.112675,0.140781,0.053557,0.265516,0.378274,0.484127,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,1,True
1,validation,MBNK,lstm,60,0.100523,0.132065,-6.077864,-0.413370,-0.396388,0.516667,month,21,limited_history,True,160,182,22,182,1260,2024-09-05,2025-04-17,1,True
2,validation,SBER,lstm,126,0.078237,0.100344,-0.514296,-0.134038,-0.124559,0.452381,month,21,mature,False,1238,1260,22,4142,1260,2019-11-08,2024-10-24,1,True
3,validation,SBERP,lstm,126,0.071503,0.097143,-0.520221,-0.057074,0.009392,0.563492,month,21,mature,False,1238,1260,22,4142,1260,2019-11-08,2024-10-24,1,True
4,validation,SVCB,lstm,80,0.062049,0.075518,-1.366177,-0.233437,-0.225809,0.412500,month,21,limited_history,True,219,241,22,241,1260,2024-04-27,2025-03-04,1,True
5,validation,T,lstm,126,0.137494,0.173093,-1.715325,-0.304818,-0.289947,0.412698,month,21,mature,False,1130,1152,22,1152,1260,2020-03-13,2024-10-22,1,True
6,validation,VTBR,lstm,126,0.127768,0.157085,-0.297985,-0.361763,-0.435315,0.404762,month,21,mature,False,1238,1260,22,4098,1260,2019-11-01,2024-10-24,1,True


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,lstm,126,0.083184,0.104529,-0.096135,0.006361,-0.099531,0.539683,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,1,True,0.484127
1,test,MBNK,lstm,60,0.086591,0.117168,-0.390413,0.549681,0.553154,0.533333,month,21,limited_history,True,220,242,22,242,1260,2024-09-05,2025-06-25,1,True,0.516667
2,test,SBER,lstm,126,0.042874,0.058081,-2.207627,-0.411589,-0.360498,0.404762,month,21,mature,False,1238,1260,22,4268,1260,2020-05-14,2025-04-11,1,True,0.452381
3,test,SBERP,lstm,126,0.047963,0.061628,-2.677400,-0.587746,-0.592585,0.396825,month,21,mature,False,1238,1260,22,4268,1260,2020-05-14,2025-04-11,1,True,0.563492
4,test,SVCB,lstm,80,0.082769,0.097270,-0.478333,-0.114033,-0.237084,0.575000,month,21,limited_history,True,299,321,22,321,1260,2024-04-27,2025-06-02,1,True,0.412500
5,test,T,lstm,126,0.052956,0.060317,-0.883572,0.076077,0.099957,0.476190,month,21,mature,False,1238,1260,22,1278,1260,2020-04-08,2025-04-09,1,True,0.412698
6,test,VTBR,lstm,126,0.092710,0.121403,-0.270523,0.424351,0.483879,0.531746,month,21,mature,False,1238,1260,22,4224,1260,2020-05-07,2025-04-11,1,True,0.404762


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning
0,-0.084907,-0.162605,0.066523,252.0,-2.444355,-3.866116,-0.133474,-1.218252,0.002646,7,CBOM,lstm,overlapping_tranches,126,False
1,0.052121,0.237871,0.037271,252.0,6.382200,12.574307,-0.018113,13.132851,0.004762,6,MBNK,lstm,overlapping_tranches,60,False
2,-0.052468,-0.102183,0.022904,252.0,-4.461264,-4.952927,-0.078066,-1.308921,0.004157,11,SBER,lstm,overlapping_tranches,126,False
3,-0.068013,-0.131400,0.022450,252.0,-5.853017,-6.984339,-0.092607,-1.418905,0.006425,17,SBERP,lstm,overlapping_tranches,126,False
4,-0.136553,-0.370287,0.053174,252.0,-6.963633,-8.722467,-0.154443,-2.397568,0.006548,11,SVCB,lstm,overlapping_tranches,80,False
5,-0.002389,-0.004773,0.027097,252.0,-0.176128,-0.189565,-0.060760,-0.078549,0.010204,27,T,lstm,overlapping_tranches,126,False
6,-0.081151,-0.155716,0.055556,252.0,-2.802876,-2.911595,-0.125272,-1.243022,0.005669,15,VTBR,lstm,overlapping_tranches,126,False
7,0.024683,0.049975,0.361228,12.0,0.138346,0.273552,-0.075375,0.663013,0.166667,1,CBOM,lstm,non_overlapping,6,True
8,0.005674,0.022890,0.016028,12.0,1.428136,NaN,-0.001499,15.271650,0.666667,2,MBNK,lstm,non_overlapping,3,True
9,-0.069335,-0.133862,0.053126,12.0,-2.519688,-3.076817,-0.069335,-1.930665,0.166667,1,SBER,lstm,non_overlapping,6,True


,check,passed,details
0,strict outer splits are available,True,split_rows=7
1,validation predictions are available,True,rows=770
2,test predictions are available,True,rows=770
3,outer split dates are chronological,True,bad_rows=0
4,train and refit windows respect max_train_rows,True,"train_over_cap=0, refit_over_cap=0"
5,validation predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
7,final refit target dates end before test starts,True,overlap_rows=0
8,final model payloads exist for test predictions,True,missing_models=0


## Merge LSTM Rows Into Main Strict Reports

In [7]:
def table_exists(path: Path) -> bool:
    return path.exists() or path.with_suffix(".csv").exists()


def load_optional_table(path: Path) -> pd.DataFrame:
    return load_table(path) if table_exists(path) else pd.DataFrame()


def drop_model_rows(frame: pd.DataFrame, model_name: str = "lstm") -> pd.DataFrame:
    if frame.empty or "model_name" not in frame.columns:
        return frame.copy()
    return frame[frame["model_name"].astype(str) != model_name].copy()


def merge_model_rows(base: pd.DataFrame, addition: pd.DataFrame, model_name: str = "lstm") -> pd.DataFrame:
    if addition.empty:
        return drop_model_rows(base, model_name)
    return pd.concat([drop_model_rows(base, model_name), addition.copy()], ignore_index=True)


def merge_best_params(base_reports: Path, lstm_reports: Path) -> list[dict]:
    base_path = base_reports / "best_params.json"
    lstm_path = lstm_reports / "best_params.json"
    base_rows = load_json(base_path) if base_path.exists() else []
    lstm_rows = load_json(lstm_path) if lstm_path.exists() else []
    merged_rows = [row for row in base_rows if row.get("model_name") != "lstm"] + lstm_rows
    save_json(merged_rows, base_path)
    table = pd.DataFrame(merged_rows)
    if not table.empty and "best_params" in table.columns:
        table["best_params"] = table["best_params"].map(lambda value: json.dumps(value, sort_keys=True))
    save_table(table, base_reports / "best_params.parquet")
    return merged_rows


def copy_lstm_payloads(lstm_root: Path, main_root: Path) -> None:
    for subdir in ["models", "hyperparams"]:
        src = lstm_root / subdir / "lstm"
        dst = main_root / subdir / "lstm"
        if src.exists():
            ensure_dir(dst.parent)
            shutil.copytree(src, dst, dirs_exist_ok=True)


def merge_lstm_into_main(item: dict, result: dict) -> dict[str, pd.DataFrame]:
    main_root = item["main_artifact_dir"] / "strict_protocol"
    main_reports = main_root / "reports"
    lstm_root = item["lstm_only_artifact_dir"] / "strict_protocol"
    lstm_reports = lstm_root / "reports"
    ensure_dir(main_reports)

    required_main = [
        "validation_predictions.parquet",
        "test_predictions.parquet",
        "test_signal_equity.parquet",
        "test_signal_metrics.parquet",
    ]
    missing = [name for name in required_main if not table_exists(main_reports / name)]
    if missing:
        raise FileNotFoundError(
            f"Cannot merge LSTM into {main_reports}; missing existing table-model reports: {missing}. "
            "Run notebooks/02_table_model_forecasting.ipynb first, or set MERGE_LSTM_INTO_MAIN_STRICT_REPORTS = False."
        )

    copy_lstm_payloads(lstm_root, main_root)

    validation_predictions = merge_model_rows(
        load_table(main_reports / "validation_predictions.parquet"),
        result["validation_predictions"],
    )
    test_predictions = merge_model_rows(
        load_table(main_reports / "test_predictions.parquet"),
        result["test_predictions"],
    )
    validation_metrics = _strict_metrics(validation_predictions, "validation")
    ranking = _validation_ranking(validation_metrics, PRIMARY_METRIC)
    selected_models = ranking[ranking["is_validation_selected"]].copy() if not ranking.empty else pd.DataFrame()
    test_metrics = _strict_metrics(test_predictions, "test")
    if not ranking.empty and not test_metrics.empty:
        enrich = ranking[["ticker", "model_name", "validation_rank", "is_validation_selected", f"validation_{PRIMARY_METRIC}"]]
        test_metrics = test_metrics.merge(enrich, on=["ticker", "model_name"], how="left")

    signal_equity = merge_model_rows(load_table(main_reports / "test_signal_equity.parquet"), result["test_signal_equity"])
    signal_metrics = merge_model_rows(load_table(main_reports / "test_signal_metrics.parquet"), result["test_signal_metrics"])
    inner_tuning_metrics = merge_model_rows(
        load_optional_table(main_reports / "inner_tuning_metrics.parquet"),
        result.get("inner_tuning_metrics", pd.DataFrame()),
    )

    outer_splits = load_table(main_reports / "outer_splits.parquet") if table_exists(main_reports / "outer_splits.parquet") else result["outer_splits"]
    leakage_audit = strict_protocol_leakage_audit(
        outer_splits=outer_splits,
        validation_predictions=validation_predictions,
        test_predictions=test_predictions,
        models_dir=main_root / "models",
    )

    save_table(outer_splits, main_reports / "outer_splits.parquet")
    save_table(validation_predictions, main_reports / "validation_predictions.parquet")
    save_table(test_predictions, main_reports / "test_predictions.parquet")
    save_table(validation_metrics, main_reports / "validation_prediction_metrics.parquet")
    save_table(ranking, main_reports / "validation_model_ranking.parquet")
    save_table(selected_models, main_reports / "selected_models_by_ticker.parquet")
    save_table(test_metrics, main_reports / "test_prediction_metrics.parquet")
    save_table(signal_equity, main_reports / "test_signal_equity.parquet")
    save_table(signal_metrics, main_reports / "test_signal_metrics.parquet")
    save_table(leakage_audit, main_reports / "leakage_audit.parquet")
    if not inner_tuning_metrics.empty:
        save_table(inner_tuning_metrics, main_reports / "inner_tuning_metrics.parquet")
    merge_best_params(main_reports, lstm_reports)

    return {
        "validation_model_ranking": ranking,
        "selected_models_by_ticker": selected_models,
        "test_prediction_metrics": test_metrics,
        "test_signal_metrics": signal_metrics,
        "leakage_audit": leakage_audit,
    }


merged_results = {}
if MERGE_LSTM_INTO_MAIN_STRICT_REPORTS:
    for item in horizon_inputs:
        horizon_name = item["horizon_name"]
        merged = merge_lstm_into_main(item, lstm_results[horizon_name])
        merged_results[horizon_name] = merged
        print(f"=== Merged LSTM into main strict reports: {horizon_name} ===")
        display(merged["validation_model_ranking"])
        display(merged["test_prediction_metrics"])
        display(merged["leakage_audit"])
        failed = merged["leakage_audit"][~merged["leakage_audit"]["passed"]]
        if not failed.empty:
            raise AssertionError(f"Merged strict leakage audit failed for {horizon_name}: {failed['check'].tolist()}")
else:
    print("Skipping merge. LSTM-only artifacts remain under artifacts/horizons/{horizon}/lstm_only/strict_protocol.")

=== Merged LSTM into main strict reports: week ===


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
4,validation,CBOM,momentum,126,0.050542,0.064387,-0.032487,0.238607,0.135598,0.603175,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,1,True
6,validation,CBOM,ridge,126,0.057839,0.074645,-0.387685,0.016420,0.056540,0.523810,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,2,False
5,validation,CBOM,naive_persistence,126,0.056416,0.070545,-0.239402,0.014613,0.006740,0.500000,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,3,False
3,validation,CBOM,lstm,126,0.050201,0.064949,-0.050582,0.017727,0.013129,0.460317,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,4,False
2,validation,CBOM,lightgbm,126,0.053503,0.068831,-0.179917,-0.093521,-0.077801,0.452381,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,5,False
1,validation,CBOM,hist_gradient_boosting,126,0.055051,0.070015,-0.220879,-0.089669,-0.068302,0.436508,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,6,False
7,validation,CBOM,xgboost,126,0.051171,0.065275,-0.061157,NaN,NaN,0.420635,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,7,False
0,validation,CBOM,catboost,126,0.052434,0.067718,-0.142078,-0.022527,-0.067333,0.412698,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,8,False
11,validation,MBNK,lstm,64,0.035874,0.047401,-1.229204,0.213218,0.205311,0.656250,week,5,limited_history,True,184,190,6,190,1260,2024-09-05,2025-05-15,1,True
14,validation,MBNK,ridge,64,0.197472,0.248458,-60.245625,0.147357,0.150504,0.640625,week,5,limited_history,True,184,190,6,190,1260,2024-09-05,2025-05-15,2,False


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,catboost,126,0.035405,0.045339,-0.177474,0.158048,0.165612,0.444444,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,8,False,0.412698
1,test,CBOM,hist_gradient_boosting,126,0.037630,0.047119,-0.271767,0.086560,0.108110,0.500000,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,6,False,0.436508
2,test,CBOM,lightgbm,126,0.034566,0.043893,-0.103584,0.095166,0.137590,0.515873,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,5,False,0.452381
3,test,CBOM,lstm,126,0.032930,0.042199,-0.020033,0.044701,0.079877,0.539683,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,4,False,0.460317
4,test,CBOM,momentum,126,0.033353,0.042393,-0.029406,-0.104179,-0.196658,0.444444,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,1,True,0.603175
5,test,CBOM,naive_persistence,126,0.035463,0.047183,-0.275220,-0.074744,-0.038584,0.500000,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,3,False,0.500000
6,test,CBOM,ridge,126,0.032458,0.042025,-0.011617,0.180101,0.217746,0.595238,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,2,False,0.523810
7,test,CBOM,xgboost,126,0.032805,0.041804,-0.001034,NaN,NaN,0.507937,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,7,False,0.420635
8,test,MBNK,catboost,64,0.027269,0.036585,0.103646,0.565771,0.588233,0.625000,week,5,limited_history,True,248,254,6,254,1260,2024-09-05,2025-07-23,6,False,0.484375
9,test,MBNK,hist_gradient_boosting,64,0.033447,0.041720,-0.165614,-0.183538,-0.163051,0.437500,week,5,limited_history,True,248,254,6,254,1260,2024-09-05,2025-07-23,7,False,0.484375


,check,passed,details
0,strict outer splits are available,True,split_rows=7
1,validation predictions are available,True,rows=6216
2,test predictions are available,True,rows=6216
3,outer split dates are chronological,True,bad_rows=0
4,train and refit windows respect max_train_rows,True,"train_over_cap=0, refit_over_cap=0"
5,validation predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
7,final refit target dates end before test starts,True,overlap_rows=0
8,final model payloads exist for test predictions,True,missing_models=0


=== Merged LSTM into main strict reports: month ===


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
6,validation,CBOM,ridge,126,0.183240,0.222773,-1.369922,0.477989,0.525369,0.714286,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,1,True
4,validation,CBOM,momentum,126,0.119886,0.145216,-0.007018,0.380199,0.360192,0.674603,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,2,False
7,validation,CBOM,xgboost,126,0.118405,0.148334,-0.050734,-0.585570,-0.444415,0.619048,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,3,False
1,validation,CBOM,hist_gradient_boosting,126,0.120147,0.149107,-0.061706,-0.573815,-0.495051,0.619048,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,4,False
2,validation,CBOM,lightgbm,126,0.121789,0.149149,-0.062307,-0.537117,-0.469035,0.619048,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,5,False
5,validation,CBOM,naive_persistence,126,0.121137,0.145796,-0.015083,0.113085,0.113995,0.563492,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,6,False
3,validation,CBOM,lstm,126,0.112675,0.140781,0.053557,0.265516,0.378274,0.484127,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,7,False
0,validation,CBOM,catboost,126,0.127382,0.155058,-0.148142,-0.262248,-0.323231,0.428571,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,8,False
9,validation,MBNK,hist_gradient_boosting,60,0.048051,0.063459,-0.634216,NaN,NaN,0.516667,month,21,limited_history,True,160,182,22,182,1260,2024-09-05,2025-04-17,1,True
14,validation,MBNK,ridge,60,0.048415,0.063871,-0.655535,-0.121510,-0.160711,0.516667,month,21,limited_history,True,160,182,22,182,1260,2024-09-05,2025-04-17,2,False


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,catboost,126,0.081123,0.097541,0.045514,0.278584,0.291801,0.650794,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,8,False,0.428571
1,test,CBOM,hist_gradient_boosting,126,0.087954,0.099400,0.008791,0.122738,0.184148,0.444444,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,4,False,0.619048
2,test,CBOM,lightgbm,126,0.085770,0.099409,0.008607,0.126055,0.130726,0.634921,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,5,False,0.619048
3,test,CBOM,lstm,126,0.083184,0.104529,-0.096135,0.006361,-0.099531,0.539683,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,7,False,0.484127
4,test,CBOM,momentum,126,0.088952,0.100350,-0.010248,-0.109474,-0.080993,0.230159,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,2,False,0.674603
5,test,CBOM,naive_persistence,126,0.086153,0.100820,-0.019735,0.059501,0.029639,0.531746,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,6,False,0.563492
6,test,CBOM,ridge,126,0.083977,0.104455,-0.094590,0.295857,0.288057,0.619048,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,1,True,0.714286
7,test,CBOM,xgboost,126,0.085955,0.098517,0.026311,0.203773,0.405381,0.761905,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,3,False,0.619048
8,test,MBNK,catboost,60,0.086232,0.099695,-0.006644,0.611376,0.607669,0.533333,month,21,limited_history,True,220,242,22,242,1260,2024-09-05,2025-06-25,5,False,0.516667
9,test,MBNK,hist_gradient_boosting,60,0.093132,0.112907,-0.291115,0.692720,0.669023,0.533333,month,21,limited_history,True,220,242,22,242,1260,2024-09-05,2025-06-25,1,True,0.516667


,check,passed,details
0,strict outer splits are available,True,split_rows=7
1,validation predictions are available,True,rows=6160
2,test predictions are available,True,rows=6160
3,outer split dates are chronological,True,bad_rows=0
4,train and refit windows respect max_train_rows,True,"train_over_cap=0, refit_over_cap=0"
5,validation predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
7,final refit target dates end before test starts,True,overlap_rows=0
8,final model payloads exist for test predictions,True,missing_models=0


## Next Step

Run `03_model_comparison.ipynb` after this notebook. If merge is enabled, it will load the main strict reports and include LSTM together with the previously trained table models.